# 05. 원곡 단위 Train·Validation·Test 분할

같은 `original_audio`에서 나온 REAL과 FAKE를 한 split에 둔다. 파일 행을 무작위로 나누면 같은 원곡이 학습과 Test에 동시에 나타날 수 있기 때문이다. 장르 분포도 함께 확인한다.

In [1]:
from pathlib import Path
import pandas as pd
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path("/Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project")

MASTER_PATH = PROJECT_ROOT / "data/metadata/master_manifest.csv"
GROUP_SPLIT_PATH = PROJECT_ROOT / "data/metadata/original_audio_split.csv"
OUTPUT_PATH = PROJECT_ROOT / "data/metadata/master_manifest_with_split.csv"

RANDOM_STATE = 42

print("MASTER_PATH      :", MASTER_PATH)
print("GROUP_SPLIT_PATH :", GROUP_SPLIT_PATH)
print("OUTPUT_PATH      :", OUTPUT_PATH)
print("RANDOM_STATE     :", RANDOM_STATE)

MASTER_PATH      : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/master_manifest.csv
GROUP_SPLIT_PATH : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/original_audio_split.csv
OUTPUT_PATH      : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/master_manifest_with_split.csv
RANDOM_STATE     : 42


## 1. Master Manifest 로드 및 기본 확인

이전 단계에서 생성한 `master_manifest.csv`가 예상 구조를 갖는지 확인한다.

In [2]:
master = pd.read_csv(MASTER_PATH)

print("===== MASTER MANIFEST =====")
print("Rows                 :", len(master))
print("Unique original_audio:", master["original_audio"].nunique())

print("\nLabel distribution:")
print(master["label"].value_counts())

print("\nGenre distribution:")
print(master["genre"].value_counts())

display(master.head())

===== MASTER MANIFEST =====
Rows                 : 3458
Unique original_audio: 296

Label distribution:
label
FAKE    3162
REAL     296
Name: count, dtype: int64

Genre distribution:
genre
Electronic    1249
Rock          1238
Pop            971
Name: count, dtype: int64


,sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,description,path_in_dataset,file_exists
0,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,NaN,NaN,True
1,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,NaN,NaN,True
2,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,NaN,NaN,True
3,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/114/114244.mp3,114244.0,NaN,NaN,True
4,sample_00004,3 am West End - statusq,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/112/112378.mp3,112378.0,NaN,NaN,True


## 2. original_audio 단위 Group Table 생성

각 `original_audio`가 하나의 장르만 갖는지 확인하고, group-level table을 만든다.

In [3]:
# original_audio 단위 Group Table 생성
group_table = (
    master.groupby("original_audio")
    .agg(
        genre=("genre", "first"),
        genre_count=("genre", "nunique"),
        total_samples=("sample_id", "size"),
        real_count=("label", lambda x: (x == "REAL").sum()),
        fake_count=("label", lambda x: (x == "FAKE").sum()),
    )
    .reset_index()
)

print("Groups:", len(group_table))
print("Groups with genre_count != 1:", int((group_table["genre_count"] != 1).sum()))
print("Groups with real_count != 1 :", int((group_table["real_count"] != 1).sum()))

print("\nGroup-level genre distribution:")
print(group_table["genre"].value_counts())

display(group_table.head())

Groups: 296
Groups with genre_count != 1: 0
Groups with real_count != 1 : 0

Group-level genre distribution:
genre
Electronic    118
Rock          111
Pop            67
Name: count, dtype: int64


,original_audio,genre,genre_count,total_samples,real_count,fake_count
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,1,6,1,5
1,1984 - Punk Rock Opera,Rock,1,14,1,13
2,2 (Wasn't There) - Isle of Pine,Rock,1,17,1,16
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,1,6,1,5
4,3 am West End - statusq,Electronic,1,6,1,5


## 3. original_audio Group을 70 / 15 / 15로 분할

먼저 Train 70%와 Temporary 30%로 나누고, Temporary를 Validation / Test로 절반씩 나눈다.

두 단계 모두 `genre`를 기준으로 stratification한다.

In [4]:
train_groups, temp_groups = train_test_split(
    group_table,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=group_table["genre"],
)

val_groups, test_groups = train_test_split(
    temp_groups,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=temp_groups["genre"],
)

train_groups = train_groups.copy()
val_groups = val_groups.copy()
test_groups = test_groups.copy()

# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
train_groups["split"] = "train"
val_groups["split"] = "val"
test_groups["split"] = "test"

group_split = pd.concat(
    [train_groups, val_groups, test_groups],
    ignore_index=True,
)

print("===== GROUP SPLIT COUNTS =====")
print(group_split["split"].value_counts())

print("\nTotal groups:", len(group_split))

===== GROUP SPLIT COUNTS =====
split
train    207
test      45
val       44
Name: count, dtype: int64

Total groups: 296


## 4. Group-level 장르 분포 확인

split별 `original_audio` 장르 분포가 유사하게 유지되는지 확인한다.

In [5]:
# Group-level 장르 분포 확인
group_genre_count = pd.crosstab(
    group_split["split"],
    group_split["genre"],
)

group_genre_ratio = pd.crosstab(
    group_split["split"],
    group_split["genre"],
    normalize="index",
).round(4)

print("===== GROUP-LEVEL GENRE COUNT =====")
display(group_genre_count)

print("===== GROUP-LEVEL GENRE RATIO =====")
display(group_genre_ratio)

===== GROUP-LEVEL GENRE COUNT =====


genre,Electronic,Pop,Rock
split,,,
test,18,10,17
train,82,47,78
val,18,10,16


===== GROUP-LEVEL GENRE RATIO =====


genre,Electronic,Pop,Rock
split,,,
test,0.4000,0.2222,0.3778
train,0.3961,0.2271,0.3768
val,0.4091,0.2273,0.3636


## 5. Master Manifest에 split 부여

`original_audio`를 기준으로 group split 정보를 3,458개 sample에 결합한다.

In [6]:
split_map = group_split[["original_audio", "split"]].copy()

master_split = master.merge(
    split_map,
    on="original_audio",
    how="left",
    validate="many_to_one",
)

print("Rows:", len(master_split))
print("Missing split:", master_split["split"].isna().sum())

print("\nSample-level split distribution:")
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
print(master_split["split"].value_counts())

display(master_split.head())

Rows: 3458
Missing split: 0

Sample-level split distribution:
split
train    2392
test      539
val       527
Name: count, dtype: int64


,sample_id,original_audio,label,label_id,source,genre,generator,audio_path,track_id,description,path_in_dataset,file_exists,split
0,sample_00000,"10,000 People Chanting, ""I'm an Individual"" - ...",REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/140/140932.mp3,140932.0,NaN,NaN,True,train
1,sample_00001,1984 - Punk Rock Opera,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/149/149410.mp3,149410.0,NaN,NaN,True,train
2,sample_00002,2 (Wasn't There) - Isle of Pine,REAL,0,FMA,Rock,NaN,data/raw/FMA/selected_30s/066/066449.mp3,66449.0,NaN,NaN,True,val
3,sample_00003,2Much (Andy Spinelli & Alex Sánchez House Edit...,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/114/114244.mp3,114244.0,NaN,NaN,True,test
4,sample_00004,3 am West End - statusq,REAL,0,FMA,Electronic,NaN,data/raw/FMA/selected_30s/112/112378.mp3,112378.0,NaN,NaN,True,test


## 6. Source-family Leakage 검증

Train / Validation / Test 사이에 동일한 `original_audio`가 하나라도 겹치면 안 된다.

In [7]:
# 같은 원곡의 표본이 여러 분할에 섞이지 않도록 저장된 split을 그대로 사용한다.
train_ids = set(group_split.loc[group_split["split"] == "train", "original_audio"])
val_ids = set(group_split.loc[group_split["split"] == "val", "original_audio"])
test_ids = set(group_split.loc[group_split["split"] == "test", "original_audio"])

train_val_overlap = train_ids & val_ids
train_test_overlap = train_ids & test_ids
val_test_overlap = val_ids & test_ids

print("===== ORIGINAL_AUDIO OVERLAP CHECK =====")
print("Train ∩ Val :", len(train_val_overlap))
print("Train ∩ Test:", len(train_test_overlap))
print("Val ∩ Test  :", len(val_test_overlap))

all_group_ids = train_ids | val_ids | test_ids

print("\nAssigned original_audio:", len(all_group_ids))
print("Expected original_audio:", master["original_audio"].nunique())

===== ORIGINAL_AUDIO OVERLAP CHECK =====
Train ∩ Val : 0
Train ∩ Test: 0
Val ∩ Test  : 0

Assigned original_audio: 296
Expected original_audio: 296


## 7. Split별 REAL / FAKE 분포 확인

원곡 단위 분할이므로 각 split의 REAL 수는 해당 split의 `original_audio` group 수와 같다.
FAKE 수는 원곡별 생성기 구성 차이에 따라 정확히 70/15/15 비율이 아닐 수 있다.

In [8]:
# Split별 REAL / FAKE 분포 확인
label_count = pd.crosstab(
    master_split["split"],
    master_split["label"],
)

print("===== LABEL DISTRIBUTION BY SPLIT =====")
display(label_count)

label_ratio = pd.crosstab(
    master_split["split"],
    master_split["label"],
    normalize="index",
).round(4)

print("===== LABEL RATIO BY SPLIT =====")
display(label_ratio)

===== LABEL DISTRIBUTION BY SPLIT =====


label,FAKE,REAL
split,,
test,494,45
train,2185,207
val,483,44


===== LABEL RATIO BY SPLIT =====


label,FAKE,REAL
split,,
test,0.9165,0.0835
train,0.9135,0.0865
val,0.9165,0.0835


## 8. Split별 장르 분포 확인

sample-level에서도 장르 분포가 지나치게 치우치지 않았는지 확인한다.

In [9]:
# Split별 장르 분포 확인
sample_genre_count = pd.crosstab(
    master_split["split"],
    master_split["genre"],
)

sample_genre_ratio = pd.crosstab(
    master_split["split"],
    master_split["genre"],
    normalize="index",
).round(4)

print("===== SAMPLE-LEVEL GENRE COUNT =====")
display(sample_genre_count)

print("===== SAMPLE-LEVEL GENRE RATIO =====")
display(sample_genre_ratio)

===== SAMPLE-LEVEL GENRE COUNT =====


genre,Electronic,Pop,Rock
split,,,
test,185,141,213
train,894,677,821
val,170,153,204


===== SAMPLE-LEVEL GENRE RATIO =====


genre,Electronic,Pop,Rock
split,,,
test,0.3432,0.2616,0.3952
train,0.3737,0.2830,0.3432
val,0.3226,0.2903,0.3871


## 9. Split별 AI Generator 분포 확인

REAL의 `generator`는 결측값이므로 제외하고, FAKE 데이터만 대상으로 생성기 분포를 확인한다.

In [10]:
# Split별 AI Generator 분포 확인
fake_only = master_split[master_split["label"] == "FAKE"].copy()

generator_count = pd.crosstab(
    fake_only["split"],
    fake_only["generator"],
)

print("===== GENERATOR DISTRIBUTION BY SPLIT =====")
display(generator_count)

===== GENERATOR DISTRIBUTION BY SPLIT =====


generator,acestep,audioldm,brev,diffrhythm,elevenlabs,mubert,musicgen,producer,songgen,stableaudio,suno,udio
split,,,,,,,,,,,,
test,45,45,48,47,48,24,45,25,45,26,48,48
train,206,203,204,207,204,100,205,102,205,141,204,204
val,43,44,46,45,48,25,43,24,42,27,48,48


## 10. 최종 Split QC

다음 조건을 모두 확인한다.

- 전체 296개 `original_audio`가 정확히 한 split에 배정됨
- split 간 `original_audio` overlap이 없음
- 모든 sample에 split이 존재함
- 전체 sample 수가 3,458개로 유지됨
- REAL 296개 / FAKE 3,162개가 유지됨

In [11]:
# 최종 Split QC
qc_summary = pd.DataFrame(
    {
        "check": [
            "total_groups",
            "train_groups",
            "val_groups",
            "test_groups",
            "train_val_overlap",
            "train_test_overlap",
            "val_test_overlap",
            "master_rows",
            "missing_split",
            "real_rows",
            "fake_rows",
        ],
        "value": [
            len(all_group_ids),
            len(train_ids),
            len(val_ids),
            len(test_ids),
            len(train_val_overlap),
            len(train_test_overlap),
            len(val_test_overlap),
            len(master_split),
            int(master_split["split"].isna().sum()),
            int((master_split["label"] == "REAL").sum()),
            int((master_split["label"] == "FAKE").sum()),
        ],
    }
)

display(qc_summary)

core_qc_pass = (
    len(all_group_ids) == 296
    and len(train_val_overlap) == 0
    and len(train_test_overlap) == 0
    and len(val_test_overlap) == 0
    and len(master_split) == 3458
    and int(master_split["split"].isna().sum()) == 0
    and int((master_split["label"] == "REAL").sum()) == 296
    and int((master_split["label"] == "FAKE").sum()) == 3162
)

print("===== FINAL RESULT =====")
print("Core QC PASS:", core_qc_pass)

,check,value
0,total_groups,296
1,train_groups,207
2,val_groups,44
3,test_groups,45
4,train_val_overlap,0
5,train_test_overlap,0
6,val_test_overlap,0
7,master_rows,3458
8,missing_split,0
9,real_rows,296


===== FINAL RESULT =====
Core QC PASS: True


## 11. Split 결과 저장

두 개의 파일을 저장한다.

1. `original_audio_split.csv`
   - 296개 원곡의 split 배정표

2. `master_manifest_with_split.csv`
   - 기존 master manifest에 `split` 컬럼을 추가한 sample-level manifest

In [12]:
if not core_qc_pass:
    raise RuntimeError(
        "Split Core QC가 통과하지 않았습니다. 저장 전에 위 결과를 확인하세요."
    )

GROUP_SPLIT_PATH.parent.mkdir(parents=True, exist_ok=True)

group_split[
    [
        "original_audio",
        "genre",
        "total_samples",
        "real_count",
        "fake_count",
        "split",
    ]
].sort_values(["split", "original_audio"]).to_csv(
    GROUP_SPLIT_PATH,
    index=False,
    encoding="utf-8-sig",
)

master_split.to_csv(
    OUTPUT_PATH,
    index=False,
    encoding="utf-8-sig",
)

print("Saved group split :", GROUP_SPLIT_PATH)
print("Saved master split:", OUTPUT_PATH)
print("Rows              :", len(master_split))

Saved group split : /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/original_audio_split.csv
Saved master split: /Users/seungjae/Desktop/SNU/기계학습 & 딥러닝/project/data/metadata/master_manifest_with_split.csv
Rows              : 3458


## 이어지는 기록

분할된 데이터의 클래스·장르·생성기 분포는 06번에서 확인한다.

## 원곡 단위 분할 결과

- `original_audio` 기준으로 Train 207개, Validation 44개, Test 45개 그룹을 분할했다.
- Sample 수는 Train 2,392개, Validation 527개, Test 539개다.
- Label 분포는 Train REAL/FAKE 207/2,185, Validation 44/483, Test 45/494다.
- 동일 `original_audio`가 여러 split에 섞이지 않음을 검증했다.
- 결과를 `original_audio_split.csv`와 `master_manifest_with_split.csv`에 저장했다.